In [1]:
#import libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow.keras import layers
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import wandb
from wandb.integration.keras import WandbCallback
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from wandb.sdk.data_types.graph import Graph



I0000 00:00:1775656027.770768  641755 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
#Les problèmes sont wandb et save best model dans le cas 2

In [11]:
# Task 1.1.1

In [2]:
early_stop = EarlyStopping(
    monitor='val_loss',   # surveille la loss sur validation
    patience=3,           # attend 3 epochs sans amélioration
    restore_best_weights=True
)

In [3]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # divise le LR par 2
    patience=2,      # après 2 epochs sans progrès
    min_lr=1e-5
)

In [4]:
checkpoint = ModelCheckpoint(
    "best_model.h5",          # nom du fichier
    monitor="val_loss",       # on surveille la validation
    save_best_only=True,      # garde seulement le meilleur
    mode="min",               # plus petite loss = mieux
    verbose=1
)

In [5]:
wandb.init(
    project="Lab1",
    name="NN_model_tf",
    config={
        "learning_rate": lr_scheduler,
        "batch_size": 32,
        "epochs": 20,
        "early_stopping": early_stop,
        "dropout": 0.5
    }
)

config = wandb.config

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
# Charger le fichier (IMPORTANT : separator = tab)
df = pd.read_csv("amazon_cells_labelled_LARGE_25K.txt", sep="\t", header=None)

# Renommer colonnes
df.columns = ["text", "label"]

# Séparer
X = df["text"]
y = df["label"]

# Train + temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Validation + test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(len(X_train), len(X_val), len(X_test))


print(X.head())
print(y.head())

17500 3750 3750
0    I've read this book with much expectation, it ...
1    love it...very touch.it's to bad that in the d...
2    The creepiest book I've ever read! It's a cree...
3    It starts off a bit slow, but once the product...
4    As good as this book may be, the print quality...
Name: text, dtype: str
0    0
1    1
2    1
3    1
4    0
Name: label, dtype: int64


In [7]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_val_vec   = vectorizer.transform(X_val).toarray()
X_test_vec  = vectorizer.transform(X_test).toarray()

In [8]:
model_ann = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(5000,)),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1775652313.925976  639992 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8254 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:21:00.0, compute capability: 7.5


In [9]:
model_ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [10]:
model_ann.fit(
    X_train_vec, y_train,
    validation_data=(X_val_vec, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop, lr_scheduler, checkpoint, WandbCallback()]
)

wandb: WARNING WandbCallback is deprecated and will be removed in a future release. Please use the WandbMetricsLogger, WandbModelCheckpoint, and WandbEvalCallback callbacks instead. See https://docs.wandb.ai/guides/integrations/keras for more information.
wandb: WARNING The save_model argument by default saves the model in the HDF5 format that cannot save custom objects like subclassed models and custom layers. This behavior will be deprecated in a future release in favor of the SavedModel format. Meanwhile, the HDF5 model is saved as W&B files and the SavedModel as W&B Artifacts.


Epoch 1/20


I0000 00:00:1775652324.217906  640340 service.cc:153] XLA service 0x7fe7100307b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775652324.217941  640340 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.3.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1775652324.399340  640340 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775652324.684056  640340 cuda_dnn.cc:461] Loaded cuDNN version 91900
I0000 00:00:1775652324.861893  640340 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1430__.10
I0000 00:00:1775652327.038416  640340 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


AttributeError: 'Node' object has no attribute 'inbound_layers'

In [10]:
best_model = load_model("best_model.h5")
y_pred = best_model.predict(X_val_vec)
y_pred = (y_pred > 0.5).astype(int)

118/118 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step


In [11]:
cm = confusion_matrix(y_val, y_pred)
print(cm)

[[1230  239]
 [ 225 2056]]


In [12]:
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.84      0.84      1469
           1       0.90      0.90      0.90      2281

    accuracy                           0.88      3750
   macro avg       0.87      0.87      0.87      3750
weighted avg       0.88      0.88      0.88      3750



In [ ]:
#Task 1.1.2

In [7]:
# Charger le fichier (IMPORTANT : separator = tab)
df = pd.read_csv("amazon_cells_labelled_LARGE_25K.txt", sep="\t", header=None)

# Renommer colonnes
df.columns = ["text", "label"]

# Séparer
X = df["text"]
y = df["label"]

# Train + temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Validation + test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
# Paramètres
MAX_VOCAB = 10000
MAX_LEN = 100  # longueur max des phrases

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)  # X_train = liste de phrases

# Séquences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

# Padding pour avoir la même longueur
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')

In [20]:
checkpoint = ModelCheckpoint(
    "best_model_LSTM.h5",          # nom du fichier
    monitor="val_loss",       # on surveille la validation
    save_best_only=True,      # garde seulement le meilleur
    mode="min",               # plus petite loss = mieux
    verbose=1
)

In [8]:
VOCAB_SIZE = MAX_VOCAB
EMBED_DIM = 128
LSTM_UNITS = 64
DROPOUT_RATE = 0.5

model_bilstm = Sequential()
model_bilstm.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM))  # plus besoin de input_length
model_bilstm.add(Bidirectional(LSTM(LSTM_UNITS)))
model_bilstm.add(Dropout(DROPOUT_RATE))
model_bilstm.add(Dense(1, activation='sigmoid'))

model_bilstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_bilstm.summary()

I0000 00:00:1775656062.515576  641755 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8254 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:21:00.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Initialise WandB
wandb.init(project="Lab1", name="BiLSTM_sentiment_analysis_TF")

# Entraînement
model_bilstm.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[
        early_stop,        # ton callback EarlyStopping
        lr_scheduler,      # ton callback LearningRateScheduler
        checkpoint,        # sauvegarde du meilleur modèle
        WandbCallback(log_model=True, save_graph=False)  # graphe désactivé ici pour éviter l'erreur
    ]
)

# Logger le graphe manuellement
graph = Graph.from_keras(model_bilstm)
wandb.run.summary["graph"] = graph

wandb: WARNING WandbCallback is deprecated and will be removed in a future release. Please use the WandbMetricsLogger, WandbModelCheckpoint, and WandbEvalCallback callbacks instead. See https://docs.wandb.ai/guides/integrations/keras for more information.
wandb: WARNING The save_model argument by default saves the model in the HDF5 format that cannot save custom objects like subclassed models and custom layers. This behavior will be deprecated in a future release in favor of the SavedModel format. Meanwhile, the HDF5 model is saved as W&B files and the SavedModel as W&B Artifacts.


Epoch 1/20


I0000 00:00:1775656078.322103  642105 cuda_dnn.cc:461] Loaded cuDNN version 91900


547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.7439 - loss: 0.4901
Epoch 1: val_loss improved from None to 0.31213, saving model to best_model.h5



Epoch 1: finished saving model to best_model.h5


ValueError: The `save_format` argument is deprecated in Keras 3. Please remove this argument and pass a file path with either `.keras` or `.h5` extension.Received: save_format=tf